# Notebook 6 – Outlier Treatment
`customer_transactions_raw.csv`.

Every flagged outlier is evaluated individually before deciding whether to remove, cap, transform, or retain it. **Outliers are not automatically removed.**

In [1]:
import pandas as pd
import numpy as np
df = pd.read_csv('customer_transactions_raw.csv')
df['age_numeric'] = pd.to_numeric(df['age'], errors='coerce')
df['purchase_amount_clean'] = df['purchase_amount'].astype(str).str.replace('$', '', regex=False)
df['purchase_amount_clean'] = pd.to_numeric(df['purchase_amount_clean'], errors='coerce')
df.shape

(1000, 14)

In [2]:
df[['age_numeric', 'annual_income', 'purchase_amount_clean', 'quantity']].describe()

,age_numeric,annual_income,purchase_amount_clean,quantity
count,947.00000,9.430000e+02,1000.000000,980.000000
mean,38.37698,3.251967e+06,238.573880,4.995918
std,14.55290,5.633967e+07,1641.123073,2.607834
min,-5.00000,-5.000000e+04,-100.000000,-1.000000
25%,30.00000,5.238569e+04,62.805000,3.000000
50%,38.00000,6.592116e+04,104.040000,5.000000
75%,45.00000,7.913065e+04,171.362500,7.000000
max,200.00000,1.000000e+09,25000.000000,9.000000


## 1. What is an Outlier?

An outlier is a data point that lies far from the bulk of the other observations in a dataset. It can be unusually high, unusually low, or structurally different from the expected pattern. Outliers are identified statistically, but *why* they exist requires domain judgment.

## 2. Outlier vs Error

Not every outlier is a mistake, and not every mistake looks like an outlier.

- **Valid extreme observation:** a real, plausible value that is simply rare (e.g. a high-net-worth customer with a very large annual income).
- **Error:** a value that is impossible or inconsistent with the real world (e.g. a negative age, a negative quantity, an income of 999,999,999).

The distinction matters because errors should usually be corrected or removed, while valid extremes often carry real business signal and should be kept or only capped.

## 3. IQR Method

The Interquartile Range (IQR) method flags a value as an outlier if it falls below `Q1 - 1.5*IQR` or above `Q3 + 1.5*IQR`.

In [4]:
def iqr_bounds(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return lower, upper
lower_income, upper_income = iqr_bounds(df['annual_income'].dropna())
lower_income, upper_income

(np.float64(12268.262500000004), np.float64(119248.08249999999))

In [5]:
income_outliers_iqr = df[(df['annual_income'] < lower_income) | (df['annual_income'] > upper_income)]
income_outliers_iqr[['customer_id', 'annual_income']].sort_values('annual_income')

,customer_id,annual_income
227,100748,-5.000000e+04
899,100779,-5.000000e+04
215,100305,3.724400e+03
457,100737,7.769630e+03
194,100366,1.203487e+05
616,100366,1.203487e+05
169,100196,1.212530e+05
387,100083,1.221488e+05
221,100893,1.261906e+05
357,100133,5.000000e+06


## 4. Z-Score Method

The Z-score method flags a value as an outlier if it lies more than a fixed number of standard deviations (commonly 3) from the mean. It works best on roughly normal distributions.

In [6]:
def zscore_outliers(series, threshold=3):
    mean = series.mean()
    std = series.std()
    z = (series - mean) / std
    return z.abs() > threshold
df['age_zscore_outlier'] = zscore_outliers(df['age_numeric'])
df[df['age_zscore_outlier'] == True][['customer_id', 'age_numeric']]

,customer_id,age_numeric
40,100873,150.0
240,100614,200.0
512,100393,200.0
889,100303,150.0


## 5. Percentile Method

The percentile method flags anything below a low percentile (e.g. 1st) or above a high percentile (e.g. 99th) as an outlier. It is less sensitive to extreme skew than the mean-based Z-score method.

In [7]:
lower_p, upper_p = df['purchase_amount_clean'].quantile([0.01, 0.99])
lower_p, upper_p

(12.2349, 427.44729999999976)

In [8]:
purchase_outliers_pct = df[(df['purchase_amount_clean'] < lower_p) | (df['purchase_amount_clean'] > upper_p)]
purchase_outliers_pct[['customer_id', 'purchase_amount_clean']].sort_values('purchase_amount_clean')

,customer_id,purchase_amount_clean
139,100770,-100.00
627,100877,-100.00
102,100912,2.73
45,100578,4.67
777,100452,8.69
141,100118,9.29
253,100659,10.25
122,100004,11.05
482,100403,11.38
169,100196,11.73


## 6. Per-Outlier Review

Each flagged value below is evaluated individually: is it an error, a valid extreme, and how should it be treated?

### 6.1 Age outliers

| customer_id | age | Error? | Valid extreme? | Remove? | Cap? | Transform? | Reasoning |
|---|---|---|---|---|---|---|---|
| 100371, 100469, 100145 | -5 | Yes | No | Yes | No | No | A negative age is physically impossible. This is a data entry error, not a rare real value, so the record should either be removed or the age re-collected/imputed rather than capped, since capping would fabricate a false minimum age. |
| 100873, 100303 | 150 | Yes | No | Yes | No | No | No human customer is 150 years old. This exceeds any plausible range and is almost certainly a typo (e.g. an extra digit) or system default. Should be treated as missing and either corrected from source or removed if unrecoverable. |
| 100614, 100393 | 200 | Yes | No | Yes | No | No | Same reasoning as above — impossible value, very likely a data entry or unit error. Removing or nulling is appropriate; capping to 100 would silently invent a value with no basis. |

**Why no capping here:** capping is appropriate for *plausible but extreme* values, not for physically impossible ones. Capping age at 100 would still leave a fabricated, unverifiable data point in the dataset.

In [9]:
invalid_age_mask = (df['age_numeric'] < 0) | (df['age_numeric'] > 100)
df.loc[invalid_age_mask, ['customer_id', 'age_numeric']]

,customer_id,age_numeric
40,100873,150.0
71,100371,-5.0
240,100614,200.0
512,100393,200.0
889,100303,150.0
976,100469,-5.0
989,100145,-5.0


### 6.2 Annual income outliers

| customer_id | annual_income | Error? | Valid extreme? | Remove? | Cap? | Transform? | Reasoning |
|---|---|---|---|---|---|---|---|
| 100748, 100779 | -50,000 | Yes | No | Yes | No | No | Income cannot be negative. This is an impossible value and should be treated as missing/corrected, not capped to zero, since the true value is unknown. |
| 100869, 100869, 100878 | 999,999,999 | Yes | No | Yes | No | No | This repeated, suspiciously round number strongly suggests a placeholder/sentinel value inserted by a broken system or blank-field default, not a real billionaire customer. Should be treated as missing and re-verified against source data. |
| 100133 | 5,000,000 | Uncertain — likely valid | Possibly | No | Yes (if kept) | Yes (for modeling) | Unlike the two cases above, this is a plausible (if rare) high income for a real individual — no repeated sentinel pattern. If retained, it should be **capped** for models sensitive to scale (e.g. linear regression) or **log-transformed** for skew reduction, rather than deleted outright, since it may reflect a genuine high-income customer. |

**Why the treatment differs within the same column:** the two 999,999,999 and -50,000 values are clear system/entry errors (impossible or implausibly repeated sentinel values), while 5,000,000 is merely rare — this is exactly the outlier-vs-error distinction from Section 2, and it means the *same column* can need different treatment for different rows.

In [10]:
error_income_mask = (df['annual_income'] < 0) | (df['annual_income'] >= 900000000)
plausible_extreme_mask = (df['annual_income'] >= 500000) & (df['annual_income'] < 900000000)

print('Likely errors (sentinel / impossible):')
print(df.loc[error_income_mask, ['customer_id', 'annual_income']])
print()
print('Plausible extreme (rare but real):')
print(df.loc[plausible_extreme_mask, ['customer_id', 'annual_income']])

Likely errors (sentinel / impossible):
     customer_id  annual_income
114       100869    999999999.0
227       100748       -50000.0
355       100869    999999999.0
560       100878    999999999.0
899       100779       -50000.0

Plausible extreme (rare but real):
     customer_id  annual_income
357       100133      5000000.0


### 6.3 Purchase amount outliers

| customer_id | purchase_amount | Error? | Valid extreme? | Remove? | Cap? | Transform? | Reasoning |
|---|---|---|---|---|---|---|---|
| 100770, 100877 | -100 | Yes | No | Yes | No | No | A negative purchase amount is impossible for a standard sale (it would represent a refund, which should be its own labeled category, not a raw transaction value). Should be corrected at the source or removed. |
| 100710, 100068, 100700, 100398 | 25,000 | Uncertain — check further | Possibly | No | Yes (if kept) | Yes (for modeling) | Repeated identical value across different customers is suspicious, but 25,000 is not impossible for a large bulk purchase — it needs cross-checking against `quantity` before deciding. If it reflects genuine bulk orders, cap for reporting/modeling rather than delete. |
| 100841 | 15,000 | Uncertain — likely valid | Possibly | No | Yes (if kept) | Yes (for modeling) | A single high but non-repeating value; more consistent with a real large transaction than a system default. Treat as a valid extreme and cap/transform rather than remove. |

**Follow-up check:** cross-referencing `purchase_amount` against `quantity` for the 25,000 rows helps decide whether they're real bulk orders (valid extremes) or system glitches (errors).

In [11]:
suspicious_high_purchase = df[df['purchase_amount_clean'] >= 15000][['customer_id', 'purchase_amount_clean', 'quantity']]
suspicious_high_purchase

,customer_id,purchase_amount_clean,quantity
70,100710,25000.0,-1.0
190,100700,25000.0,9.0
524,100068,25000.0,3.0
599,100398,25000.0,7.0
641,100841,15000.0,9.0


### 6.4 Quantity outliers

| customer_id | quantity | Error? | Valid extreme? | Remove? | Cap? | Transform? | Reasoning |
|---|---|---|---|---|---|---|---|
| 100465, 100710, 100178, 100146, 100657 | -1 | Yes | No | Yes | No | No | A negative quantity purchased is not physically meaningful in this context (it is not flagged elsewhere as a return/refund column). This is a data entry error and should be corrected or removed, not floored to 0, since a 0-quantity transaction is also meaningless for a purchase record. |

**Why not floor to 0 or 1:** flooring assumes the "true" value is close to the boundary, but here the negative sign is simply invalid — there's no principled floor value to assign without more information, so removal/correction is safer than fabricating a replacement.

In [12]:
df[df['quantity'] < 0][['customer_id', 'quantity', 'purchase_amount_clean']]

,customer_id,quantity,purchase_amount_clean
66,100465,-1.0,47.65
70,100710,-1.0,25000.00
217,100178,-1.0,118.81
708,100146,-1.0,76.68
780,100657,-1.0,193.51


## 7. Winsorization

Winsorization replaces extreme values with the nearest acceptable percentile value instead of deleting them, preserving the row count while limiting the influence of extremes.

In [15]:
from scipy.stats.mstats import winsorize
income_valid = df.loc[~error_income_mask, 'annual_income'].dropna()
winsorized_income = winsorize(income_valid, limits=[0.01, 0.01])
pd.Series(winsorized_income).describe()

C:\Users\hemak\AppData\Local\Programs\Python\Python314\Lib\site-packages\numpy\lib\_function_base_impl.py:4798: UserWarning: Warning: 'partition' will ignore the 'mask' of the MaskedArray.
  arr.partition(


count       938.000000
mean      65928.737377
std       19949.440711
min       20143.450000
25%       52455.725000
50%       65902.750000
75%       79076.985000
max      116450.350000
dtype: float64

## 8. Capping

Capping sets any value above (or below) a chosen threshold to that threshold, rather than removing the row. This is appropriate for **valid extreme values** we want to keep in the dataset but whose scale should be limited (e.g. for models sensitive to large values).

In [16]:
def cap_series(series, lower_pct=0.01, upper_pct=0.99):
    lower = series.quantile(lower_pct)
    upper = series.quantile(upper_pct)
    return series.clip(lower=lower, upper=upper)
df['purchase_amount_capped'] = cap_series(df['purchase_amount_clean'])
df[['purchase_amount_clean', 'purchase_amount_capped']].describe()

,purchase_amount_clean,purchase_amount_capped
count,1000.000000,1000.000000
mean,238.573880,125.726212
std,1641.123073,85.947171
min,-100.000000,12.234900
25%,62.805000,62.805000
50%,104.040000,104.040000
75%,171.362500,171.362500
max,25000.000000,427.447300


## 9. Flooring

Flooring is the mirror of capping: values below a chosen lower threshold are raised to that threshold. It is only appropriate when the true minimum is known and low values are valid-but-extreme rather than erroneous (unlike the negative quantity/age/income errors identified above, which should not simply be floored to a plausible minimum).

In [17]:
def floor_series(series, lower_pct=0.01):
    lower = series.quantile(lower_pct)
    return series.clip(lower=lower)
valid_quantity = df.loc[df['quantity'] >= 0, 'quantity']
floored_quantity = floor_series(valid_quantity)
floored_quantity.describe()

count    975.000000
mean       5.026667
std        2.578801
min        1.000000
25%        3.000000
50%        5.000000
75%        7.000000
max        9.000000
Name: quantity, dtype: float64

## 10. Transformation

Transformations (e.g. log, square root) reduce the influence of extreme values by compressing the scale of a skewed distribution, without discarding any data points. This is often preferable to capping when the extreme values are genuine and their relative ordering should be preserved.

In [15]:
income_for_log = df.loc[~error_income_mask, 'annual_income'].dropna()
log_income = np.log1p(income_for_log)
log_income.describe()

count    938.000000
mean      11.043664
std        0.388904
min        8.222930
25%       10.867743
50%       11.095951
75%       11.278189
max       15.424949
Name: annual_income, dtype: float64

## 11. Removing Outliers

Removal is reserved for values identified as **errors** — impossible or clearly corrupted values that cannot be trusted or meaningfully corrected (Sections 6.1, parts of 6.2, and 6.4 above).

In [18]:
rows_to_remove = (
    invalid_age_mask
    | error_income_mask
    | (df['purchase_amount_clean'] < 0)
    | (df['quantity'] < 0)
)
print('Rows flagged as errors for removal:', rows_to_remove.sum())
df_cleaned = df[~rows_to_remove].copy()
df_cleaned.shape

Rows flagged as errors for removal: 19


(981, 16)

## 12. Retaining Outliers

Values identified as **valid extreme observations** (Sections 6.2's 5,000,000 income and 6.3's 15,000/25,000 purchases) are retained in the dataset, since deleting them would remove genuine business signal. They are instead candidates for capping or transformation depending on the downstream use case (reporting vs. statistical modeling).

In [17]:
df_cleaned.to_csv('customer_transactions_outlier_reviewed.csv', index=False)
df_cleaned.shape

(981, 16)